[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/40_linear_regression.ipynb)

# 🟡 Medium: Linear Regression

Implement **linear regression** using three different approaches — all in pure PyTorch.

Given data `X` of shape `(N, D)` and targets `y` of shape `(N,)`, find weight `w` of shape `(D,)` and bias `b` (scalar) such that:

$$\hat{y} = Xw + b$$

### Signature
```python
class LinearRegression:
    def closed_form(self, X: Tensor, y: Tensor) -> tuple[Tensor, Tensor]: ...
    def gradient_descent(self, X: Tensor, y: Tensor, lr=0.01, steps=1000) -> tuple[Tensor, Tensor]: ...
    def nn_linear(self, X: Tensor, y: Tensor, lr=0.01, steps=1000) -> tuple[Tensor, Tensor]: ...
```

All methods return `(w, b)` where `w` has shape `(D,)` and `b` has shape `()`.

### Method 1 — Closed-Form (Normal Equation)
Augment X with a ones column, then solve:

$$\theta = (X_{aug}^T X_{aug})^{-1} X_{aug}^T y$$

Or use `torch.linalg.lstsq` / `torch.linalg.solve`.

### Method 2 — Gradient Descent from Scratch
Initialize `w` and `b` to zeros. Repeat for `steps` iterations:
```
pred = X @ w + b
error = pred - y
grad_w = (2/N) * X^T @ error
grad_b = (2/N) * error.sum()
w -= lr * grad_w
b -= lr * grad_b
```

### Method 3 — PyTorch nn.Linear
Create `nn.Linear(D, 1)`, use `nn.MSELoss` and an optimizer (e.g., `torch.optim.SGD`).
After training, extract `w` and `b` from the layer.

### Rules
- All inputs and outputs must be **PyTorch tensors**
- Do **NOT** use numpy or sklearn
- `closed_form` must not use iterative optimization
- `gradient_descent` must manually compute gradients (no `autograd`)
- `nn_linear` should use `torch.nn.Linear` and `loss.backward()`

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import torch.nn as nn

In [3]:
# ✏️ YOUR IMPLEMENTATION HERE

class LinearRegression:
    def closed_form(self, X: torch.Tensor, y: torch.Tensor):
        """Normal equation: w = (X^T X)^{-1} X^T y"""

        # Make y a 1D tensor: (N,)
        y = y.reshape(-1)

        # Append a column of ones to X for the bias term
        ones = torch.ones(X.shape[0], 1, dtype=X.dtype, device=X.device)
        X_aug = torch.cat([X, ones], dim=1)

        # Solve X_aug @ theta = y using least squares
        theta = torch.linalg.lstsq(X_aug, y).solution

        # Separate weights and bias
        w = theta[:-1]
        b = theta[-1]

        return w, b


    def gradient_descent(self, X: torch.Tensor, y: torch.Tensor,
                         lr: float = 0.01, steps: int = 1000):
        """Manual gradient descent loop"""

        y = y.reshape(-1)

        N, D = X.shape

        # Initialize parameters
        w = torch.zeros(D, dtype=X.dtype, device=X.device)
        b = torch.zeros((), dtype=X.dtype, device=X.device)

        for _ in range(steps):

            # 1. Forward pass
            pred = X @ w + b

            # 2. Compute error
            error = pred - y

            # 3. Compute gradients manually
            grad_w = (2 / N) * (X.T @ error)
            grad_b = (2 / N) * error.sum()

            # 4. Update parameters
            w -= lr * grad_w
            b -= lr * grad_b

        return w, b


    def nn_linear(self, X: torch.Tensor, y: torch.Tensor,
                  lr: float = 0.01, steps: int = 1000):
        """Train nn.Linear with autograd"""

        y = y.reshape(-1, 1)

        N, D = X.shape

        # Define the model
        model = nn.Linear(D, 1).to(
            device=X.device, dtype=X.dtype
        )

        # Define loss and optimizer
        criterion = nn.MSELoss()
        optimizer = torch.optim.SGD(model.parameters(), lr=lr)

        for _ in range(steps):

            # 1. Forward pass
            pred = model(X)

            # 2. Compute loss
            loss = criterion(pred, y)

            # 3. Backpropagation
            optimizer.zero_grad()
            loss.backward()

            # 4. Update parameters
            optimizer.step()

        # Extract learned parameters
        w = model.weight.detach().squeeze(0)
        b = model.bias.detach().squeeze()

        return w, b


In [4]:
# 🧪 Debug
torch.manual_seed(42)
X = torch.randn(100, 3)
true_w = torch.tensor([2.0, -1.0, 0.5])
y = X @ true_w + 3.0

model = LinearRegression()

w_cf, b_cf = model.closed_form(X, y)
print(f"Closed-form:  w={w_cf}, b={b_cf.item():.4f}")

w_gd, b_gd = model.gradient_descent(X, y, lr=0.05, steps=2000)
print(f"Grad descent: w={w_gd}, b={b_gd.item():.4f}")

w_nn, b_nn = model.nn_linear(X, y, lr=0.05, steps=2000)
print(f"nn.Linear:    w={w_nn}, b={b_nn.item():.4f}")

print(f"\nTrue:         w={true_w}, b=3.0")

Closed-form:  w=tensor([ 2.0000, -1.0000,  0.5000]), b=3.0000
Grad descent: w=tensor([ 2.0000, -1.0000,  0.5000]), b=3.0000
nn.Linear:    w=tensor([ 2.0000, -1.0000,  0.5000]), b=3.0000

True:         w=tensor([ 2.0000, -1.0000,  0.5000]), b=3.0


In [5]:
# ✅ SUBMIT
from torch_judge import check
check("linear_regression")


🧪 Testing: Linear Regression (Medium)
──────────────────────────────────────────────────
  ✅ [1/6] Closed-form returns correct shapes (8.4ms)
  ✅ [2/6] Closed-form finds correct weights (10.3ms)
  ✅ [3/6] Gradient descent converges (156.6ms)
  ✅ [4/6] nn.Linear approach works (984.0ms)
  ✅ [5/6] All three methods agree (1850.1ms)
  ✅ [6/6] Closed-form uses no autograd (1.1ms)
──────────────────────────────────────────────────
  🎉 All 6 tests passed! (3010.5ms total)
  Progress saved. Run status() to see your dashboard.

